In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import pprint

In [3]:
from eml_transformer.ingestion.sources.iem_afos import IEMAFOSSource

source = IEMAFOSSource(
    product_types=["SPS", ],
    wfos=["IND"],
    limit=3, 
)

records = source.fetch_records(
    from_date="2024-01-01",
    to_date="2025-01-05",
)

In [4]:
print("RAW RESPONSES:", len(records))
print("RAW KEYS:", records[0].keys())

RAW RESPONSES: 3
RAW KEYS: dict_keys(['source_id', 'pil', 'raw_text', 'header', 'issued_at_text', 'published_at'])


In [5]:
records

[{'source_id': '26b08e9e0aa2244bbd83cf90743653afdb62d55de5d17915c0a7cbdf5de5fa08',
  'pil': 'SPSIND',
  'raw_text': '236 \nWWUS83 KIND 021957\nSPSIND\n\nSpecial Weather Statement\nNational Weather Service Indianapolis IN\n257 PM EST Thu Jan 2 2025\n\nINZ021-028>031-035>049-054>057-063>065-072-031000-\nCarroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-\nBoone-Tipton-Hamilton-Madison-Delaware-Randolph-Vermillion-Parke-\nPutnam-Hendricks-Marion-Hancock-Henry-Morgan-Johnson-Shelby-Rush-\nBrown-Bartholomew-Decatur-Jennings-\nIncluding the cities of Delphi, Flora, Williamsport, \nWest Lebanon, Lafayette, West Lafayette, Frankfort, Kokomo, \nAttica, Covington, Veedersburg, Crawfordsville, Lebanon, \nZionsville, Tipton, Fishers, Carmel, Noblesville, Anderson, \nMuncie, Winchester, Union City, Farmland, Parker City, Clinton, \nFairview Park, Rockville, Montezuma, Rosedale, Greencastle, \nPlainfield, Brownsburg, Danville, Indianapolis, Greenfield, \nNew Castle, Martinsville, Mooresvil

In [6]:
print("PARSED RECORDS:", len(records))

for i, record in enumerate(records):
    print("\n" + "=" * 80)
    print(f"RECORD {i + 1}")
    print("=" * 80)

    raw_text = record["raw_text"]
    header = source._parse_header(raw_text)
    sections = source._parse_sections(raw_text)
    issued = source._extract_issued_text(raw_text)

    print("PIL:", record["pil"])
    print("HEADER:", header)
    print("ISSUED:", issued)
    print("SECTIONS:", list(sections.keys()))
    print("RAW LEN:", len(raw_text))

    print("\nFIRST 10 LINES:")
    print("\n".join(raw_text.splitlines()[:10]))

    print("\nKEY MESSAGES:")
    print(sections.get("KEY MESSAGES", "MISSING")[:1000])

PARSED RECORDS: 3

RECORD 1
PIL: SPSIND
HEADER: {'raw_id': '236', 'wmo': 'WWUS83', 'wmo_header': 'WWUS83 KIND 021957', 'office': 'KIND', 'issued_code': '021957', 'pil': 'SPSIND'}
ISSUED: 257 PM EST Thu Jan 2 2025
SECTIONS: []
RAW LEN: 1704

FIRST 10 LINES:
236 
WWUS83 KIND 021957
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
257 PM EST Thu Jan 2 2025

INZ021-028>031-035>049-054>057-063>065-072-031000-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-

KEY MESSAGES:
MISSING

RECORD 2
PIL: SPSIND
HEADER: {'raw_id': '231', 'wmo': 'WWUS83', 'wmo_header': 'WWUS83 KIND 020853', 'office': 'KIND', 'issued_code': '020853', 'pil': 'SPSIND'}
ISSUED: 353 AM EST Thu Jan 2 2025
SECTIONS: []
RAW LEN: 2347

FIRST 10 LINES:
231 
WWUS83 KIND 020853
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
353 AM EST Thu Jan 2 2025

INZ021-028>031-035>049-051>057-060>065-067>072-022130-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-

In [7]:
standardized = [
    source.standardize_record(record)
    for record in records
]

for i, record in enumerate(standardized):
    print("\n" + "=" * 80)
    print(f"STANDARDIZED {i + 1}")
    print("=" * 80)

    print("record_id:", record.record_id)
    print("title:", record.title)
    print("published_at:", record.published_at)
    print("region:", record.region)
    print("categories:", record.categories)
    print("text len:", len(record.text or ""))
    print("raw sections:", list(record.metadata.get("sections", {}).keys()))

    print("\nTEXT PREVIEW:")
    print((record.text or "")[:1000])


STANDARDIZED 1
record_id: 733eeca6e0b55f9ee4d78651c29c84f2fe9f4fcb62ba6bb70bae7dfcd2d047a9
title: SPS | KIND | 257 PM EST Thu Jan 2 2025
published_at: 2025-01-02T19:57:00+00:00
region: IND
categories: ['weather', 'nws', 'iem', 'afos', 'sps']
text len: 1704
raw sections: []

TEXT PREVIEW:
236 
WWUS83 KIND 021957
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
257 PM EST Thu Jan 2 2025

INZ021-028>031-035>049-054>057-063>065-072-031000-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-
Boone-Tipton-Hamilton-Madison-Delaware-Randolph-Vermillion-Parke-
Putnam-Hendricks-Marion-Hancock-Henry-Morgan-Johnson-Shelby-Rush-
Brown-Bartholomew-Decatur-Jennings-
Including the cities of Delphi, Flora, Williamsport, 
West Lebanon, Lafayette, West Lafayette, Frankfort, Kokomo, 
Attica, Covington, Veedersburg, Crawfordsville, Lebanon, 
Zionsville, Tipton, Fishers, Carmel, Noblesville, Anderson, 
Muncie, Winchester, Union City, Farmland, Parker City, Clinton, 
Fair

### Checking Stored

In [8]:
import json

path = "../data/bronze/source=iem_afos/records.jsonl"

with open(path, "r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("records loaded:", len(records))

standardized = [
    source.standardize_record(record['raw'])
    for record in records
]

for i, record in enumerate(standardized):
    # print("\n" + "=" * 80)
    # print(f"STANDARDIZED {i + 1}")
    # print("=" * 80)

    # print("record_id:", record.record_id)
    # print("title:", record.title)
    # print("published_at:", record.published_at)
    # print("region:", record.region)
    # print("categories:", record.categories)
    # print("text len:", len(record.text or ""))

    sections = record.metadata.get("sections", {}) if record.metadata else {}
    print("raw sections:", list(sections.keys()))

    # print("\nTEXT PREVIEW:")
    # print((record.text or "")[:1000])

records loaded: 39820
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
r

In [9]:
df = pd.read_parquet('../data/silver/source=iem_afos/records.parquet')

In [10]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata', 'raw'],
      dtype='object')

In [11]:
row = df.iloc[0]

for col in df.columns:
    print("\n" + "=" * 80)
    print(col.upper())
    print("=" * 80)
    print(row[col])


RECORD_ID
4b8ac526c396cf72436a0d18c06354726d8985c8ec407e28e5ac0993985f656b

SOURCE
iem_afos

SOURCE_TYPE
api

TITLE
AFD | KIWX | Issued at 208 PM EST Sat Nov 30 2024

TEXT
- Scattered lake effect snow showers over southern Michigan will become more numerous into Monday as they spread south into northern Indiana. - Unseasonable cold continues through the week with highs ranging from the upper 20s to low 30s with lows in the teens.

PUBLISHED_AT
2024-11-30T19:08:00+00:00

RETRIEVED_AT
2026-06-22T13:16:21.485782+00:00

URL
https://mesonet.agron.iastate.edu/cgi-bin/afos/retrieve.py

REGION
IWX

CATEGORIES
['weather' 'nws' 'iem' 'afos' 'afd']

METADATA
{'pil': 'AFDIWX', 'product_type': 'AFD', 'office': 'KIWX', 'sections': {'KEY MESSAGES': '- Scattered lake effect snow showers over southern Michigan will\n  become more numerous into Monday as they spread south into\n  northern Indiana.\n\n- Unseasonable cold continues through the week with highs\n  ranging from the upper 20s to low 30s with

In [12]:
import json

In [13]:
for i, row in df.head(5).iterrows():
    print("\n" + "#" * 100)

    print("TITLE:")
    print(row["title"])

    print("\nPUBLISHED_AT:")
    print(row["published_at"])

    print("\nREGION:")
    print(row["region"])

    print("\nTEXT PREVIEW:")
    print(row["text"][:1000])

    print("\nTEXT LENGTH:")
    print(len(row["text"]))


    print("\nSECTION KEYS:")
    print(row["metadata"].get("sections", {}).keys())


####################################################################################################
TITLE:
AFD | KIWX | Issued at 208 PM EST Sat Nov 30 2024

PUBLISHED_AT:
2024-11-30T19:08:00+00:00

REGION:
IWX

TEXT PREVIEW:
- Scattered lake effect snow showers over southern Michigan will become more numerous into Monday as they spread south into northern Indiana. - Unseasonable cold continues through the week with highs ranging from the upper 20s to low 30s with lows in the teens.

TEXT LENGTH:
261

SECTION KEYS:
dict_keys(['KEY MESSAGES', 'DISCUSSION', 'AVIATION /06Z TAFS THROUGH 06Z MONDAY/', 'IWX WATCHES/WARNINGS/ADVISORIES', 'SHORT TERM /THROUGH SUNDAY/', 'LONG TERM /SUNDAY NIGHT THROUGH SATURDAY/', 'DVN WATCHES/WARNINGS/ADVISORIES', 'AVIATION /00Z TAFS THROUGH 00Z MONDAY/', 'AVIATION /06Z TAFS THROUGH 12Z MONDAY/', 'LOT WATCHES/WARNINGS/ADVISORIES', 'DMX WATCHES/WARNINGS/ADVISORIES', 'SHORT TERM', 'LONG TERM', 'AVIATION', 'LSX WATCHES/WARNINGS/ADVISORIES', 'UPDATE', 'ILX WATCH

In [14]:
print(df.isna().sum())

record_id       0
source          0
source_type     0
title           0
text            0
published_at    0
retrieved_at    0
url             0
region          0
categories      0
metadata        0
raw             0
dtype: int64


In [15]:
df["published_at"].head(20)

0     2024-11-30T19:08:00+00:00
1     2024-11-30T19:25:00+00:00
2     2024-11-30T19:25:00+00:00
3     2024-11-30T19:35:00+00:00
4     2024-11-30T19:50:00+00:00
5     2024-11-30T19:50:00+00:00
6     2024-12-01T01:03:00+00:00
7     2024-12-01T01:04:00+00:00
8     2024-12-01T01:04:00+00:00
9     2024-12-01T01:05:00+00:00
10    2024-12-01T01:06:00+00:00
11    2024-12-01T01:07:00+00:00
12    2024-12-01T01:08:00+00:00
13    2024-12-01T01:09:00+00:00
14    2024-12-01T01:11:00+00:00
15    2024-12-01T01:11:00+00:00
16    2024-12-01T01:14:00+00:00
17    2024-12-01T01:14:00+00:00
18    2024-12-01T01:28:00+00:00
19    2024-12-01T01:29:00+00:00
Name: published_at, dtype: object

In [16]:
df.applymap(type).head(1).T

/tmp/ipykernel_43649/4094605693.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df.applymap(type).head(1).T


,0
record_id,<class 'str'>
source,<class 'str'>
source_type,<class 'str'>
title,<class 'str'>
text,<class 'str'>
published_at,<class 'str'>
retrieved_at,<class 'str'>
url,<class 'str'>
region,<class 'str'>
categories,<class 'numpy.ndarray'>


In [17]:
df["metadata"].apply(
    lambda r: list(r.get("sections", {}).keys())
).value_counts()

metadata
[KEY MESSAGES, DISCUSSION, AVIATION /06Z TAFS THROUGH 06Z MONDAY/, IWX WATCHES/WARNINGS/ADVISORIES, SHORT TERM /THROUGH SUNDAY/, LONG TERM /SUNDAY NIGHT THROUGH SATURDAY/, DVN WATCHES/WARNINGS/ADVISORIES, AVIATION /00Z TAFS THROUGH 00Z MONDAY/, AVIATION /06Z TAFS THROUGH 12Z MONDAY/, LOT WATCHES/WARNINGS/ADVISORIES, DMX WATCHES/WARNINGS/ADVISORIES, SHORT TERM, LONG TERM, AVIATION, LSX WATCHES/WARNINGS/ADVISORIES, UPDATE, ILX WATCHES/WARNINGS/ADVISORIES, FORECAST UPDATE, IND WATCHES/WARNINGS/ADVISORIES, DAY ONE, DAYS TWO THROUGH SEVEN, SPOTTER INFORMATION STATEMENT, SHORT TERM /THROUGH TONIGHT/, LONG TERM /MONDAY THROUGH SATURDAY/, AVIATION /18Z TAFS THROUGH 00Z TUESDAY/, AVIATION /18Z TAFS THROUGH 18Z MONDAY/, AVIATION /12Z TAFS THROUGH 18Z MONDAY/, AVIATION /12Z TAFS THROUGH 12Z MONDAY/, AVIATION /00Z TAFS THROUGH 06Z TUESDAY/, AVIATION /06Z TAFS THROUGH 12Z TUESDAY/, SHORT TERM /THROUGH MONDAY/, LONG TERM /MONDAY NIGHT THROUGH SUNDAY/, AVIATION /00Z TAFS THROUGH 00Z TUESDAY/

In [19]:
min(df['published_at'])

'2024-11-30T19:08:00+00:00'

In [20]:
max(df['published_at'])

'2026-06-18T17:30:00+00:00'

In [21]:
df["published_at"] = pd.to_datetime(df["published_at"])

date_range = pd.date_range(
    start=df["published_at"].min().floor("D"),
    end=df["published_at"].max().floor("D"),
    freq="D",
)

observed_dates = df["published_at"].dt.floor("D").unique()

missing_dates = date_range.difference(observed_dates)

print(f"Missing days: {len(missing_dates)}")
print(missing_dates[:20])

Missing days: 13
DatetimeIndex(['2025-01-30 00:00:00+00:00', '2025-03-01 00:00:00+00:00',
               '2025-03-31 00:00:00+00:00', '2025-04-30 00:00:00+00:00',
               '2025-05-30 00:00:00+00:00', '2025-06-29 00:00:00+00:00',
               '2025-07-29 00:00:00+00:00', '2025-08-28 00:00:00+00:00',
               '2025-09-27 00:00:00+00:00', '2025-10-27 00:00:00+00:00',
               '2025-11-26 00:00:00+00:00', '2025-12-26 00:00:00+00:00',
               '2026-01-25 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', freq=None)
